# ProtIntel — GPU Training Notebook

**ESM-2 → CNN + BiLSTM + Multi-Head Attention → Q3/Q8 Secondary Structure Prediction**

## Setup Instructions
1. **Runtime → Change runtime type → T4 GPU** (or better)
2. Upload `protintel_colab_package.zip` to `/content/` via Files panel (left sidebar → Upload)
3. Upload this notebook if not already open
4. **Run cells 1–10 in order — do not skip any cell**

## Cell-by-Cell Summary
| Cell | Description |
|------|-------------|
| 1 | GPU check — fails fast if no GPU detected |
| 2 | Install missing Python dependencies |
| 3 | Unpack zip → `/content/protintel/`, verify project structure |
| 4 | **Generate ESM-2 embeddings on GPU** (all 6,043 sequences, ~5 min on T4) |
| 5 | Spot-check 3 random cached embeddings (shape, NaN, non-zero) |
| 6 | Launch TensorBoard (refresh every few minutes during training) |
| 7 | Run `train.py` — 30 epochs max, AMP, EarlyStopping patience=15, ModelCheckpoint top-k=3 |
| 8 | Verify best checkpoint (epoch, Q3/Q8 accuracy, optimizer state present) |
| 9 | Zip checkpoints + TensorBoard logs → `/content/protintel_trained_outputs.zip` |
| 10 | Auto-download `protintel_trained_outputs.zip` |

In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────────────────
import subprocess, sys

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU detected — change runtime type!')

import torch
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    raise RuntimeError('GPU not available. Set Runtime → Change runtime type → T4 GPU and re-run.')

In [ ]:
# ── Cell 2: Install dependencies ──────────────────────────────────────────────
# Colab already ships with torch/numpy; install the missing packages only.
import subprocess, sys

result = subprocess.run(
    [
        sys.executable, '-m', 'pip', 'install', '-q',
        'transformers>=4.36.0',
        'fair-esm>=2.0.0',
        'pyyaml>=6.0',
        'pydantic>=2.5.0',
        'pydantic-settings>=2.1.0',
        'torchmetrics>=1.2.0',
        'scikit-learn>=1.3.0',
        'tensorboard>=2.15.0',
        'tqdm>=4.66.0',
        'biopython>=1.82',
    ],
    capture_output=True, text=True
)
if result.returncode != 0:
    print(result.stdout)
    print(result.stderr)
    raise RuntimeError(f'pip install failed with code {result.returncode}')
print('✓ Dependencies installed')

In [ ]:
# ── Cell 3: Unpack the project package ────────────────────────────────────────
import zipfile, os
from pathlib import Path

PACKAGE_NAME = 'protintel_colab_package.zip'
INSTALL_DIR  = Path('/content/protintel')

# Locate the zip (uploaded to /content/ by the user)
zip_path = Path(f'/content/{PACKAGE_NAME}')
if not zip_path.exists():
    raise FileNotFoundError(
        f'{zip_path} not found.\n'
        'Upload protintel_colab_package.zip via the Files panel (left sidebar → Upload) '
        'before running this cell.'
    )

# Unzip (idempotent — skip if already unpacked)
if not INSTALL_DIR.exists():
    print(f'Unpacking {PACKAGE_NAME} ...')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('/content')
    print(f'✓ Unpacked to {INSTALL_DIR}')
else:
    print(f'✓ Already unpacked at {INSTALL_DIR}')

# Set working directory and Python path
os.chdir(INSTALL_DIR)
sys.path.insert(0, str(INSTALL_DIR))
print(f'Working directory: {os.getcwd()}')

# Verify key project structure (embeddings dir may be empty at this point — that's OK)
required_paths = [
    'train.py',
    'evaluate.py',
    'scripts/generate_embeddings.py',
    'src/models/protintel_model.py',
    'src/data/protein_dataset.py',
    'datasets/raw',
    'configs/training.yaml',
    'configs/model.yaml',
    'configs/data.yaml',
]
all_present = True
for p in required_paths:
    exists = Path(p).exists()
    if not exists:
        all_present = False
    print(f'  {"✓" if exists else "✗ MISSING"}  {p}')

if not all_present:
    raise RuntimeError('One or more required paths are missing — check zip contents.')

print('\n✓ Project structure verified.')

# Embeddings dir (will be populated by Cell 4)
emb_dir = INSTALL_DIR / 'datasets' / 'processed' / 'embeddings'
emb_dir.mkdir(parents=True, exist_ok=True)
pt_files = list(emb_dir.glob('*.pt'))
print(f'Cached embeddings at this point: {len(pt_files)} files (will be generated in Cell 4)')

In [ ]:
# ── Cell 4: Generate ESM-2 embeddings on GPU ──────────────────────────────────
# Runs scripts/generate_embeddings.py for ALL sequences (CullPDB train + CB513 test).
# Uses SHA-256 hash-based caching — already-cached sequences are skipped automatically.
# Expected: ~5,531 CullPDB + ~512 CB513 = ~6,043 total unique sequences.
# Expected time: ~5 minutes on T4 GPU (vs ~10 min CPU in the prior local run).
import subprocess, sys, time

cmd = [
    sys.executable, 'scripts/generate_embeddings.py',
    '--device', 'cuda',
    '--batch-size', '16',   # T4 can handle larger batches than CPU
]

print('Running:', ' '.join(cmd))
print('=' * 60)
t0 = time.time()

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()

elapsed = time.time() - t0
if proc.returncode not in (0, 1):
    # Exit code 1 can be a tqdm/stderr redirect artifact — only fail on 2+
    raise RuntimeError(f'generate_embeddings.py exited with code {proc.returncode}')

from pathlib import Path
emb_dir = Path('datasets/processed/embeddings')
pt_files = list(emb_dir.glob('*.pt'))
print(f'\n{"=" * 60}')
print(f'✓ Embedding generation complete!')
print(f'  Cached .pt files : {len(pt_files)}')
print(f'  Wall-clock time  : {elapsed:.0f}s ({elapsed/60:.1f} min)')

if len(pt_files) < 100:
    raise RuntimeError(
        f'Only {len(pt_files)} embedding files found after generation — '
        'something went wrong. Check output above for errors.'
    )
print(f'\n✓ Sufficient embeddings present. Proceeding to spot-check (Cell 5).')

In [ ]:
# ── Cell 5: Spot-check 3 random cached embeddings ─────────────────────────────
import random, torch
from pathlib import Path

emb_dir = Path('datasets/processed/embeddings')
pt_files = list(emb_dir.glob('*.pt'))
assert len(pt_files) >= 3, f'Need at least 3 embeddings for spot-check, found {len(pt_files)}'

random.seed(42)
sample_files = random.sample(pt_files, 3)
all_ok = True

print(f'Spot-checking 3 of {len(pt_files)} cached embeddings...')
for f in sample_files:
    emb = torch.load(f, map_location='cpu', weights_only=True)
    L, D = emb.shape
    has_nan = torch.isnan(emb).any().item()
    all_zero = (emb == 0).all().item()
    ok = (D == 480 and not has_nan and not all_zero)
    if not ok:
        all_ok = False
    status = 'OK' if ok else 'FAIL'
    print(f'  {f.name[:24]}... | shape=({L}, {D}) | nan={has_nan} | all_zero={all_zero} | [{status}]')

assert all_ok, 'Embedding spot-check FAILED — check embedding generation output in Cell 4'
print('\n✓ All spot-checks passed — embeddings are valid.')

In [ ]:
# ── Cell 6: Launch TensorBoard ────────────────────────────────────────────────
# Run this cell, then start training in Cell 7.
# Refresh the TensorBoard panel every few minutes to see live metrics.
%load_ext tensorboard
%tensorboard --logdir logs/

In [ ]:
# ── Cell 7: Run full training ──────────────────────────────────────────────────
# Config (configs/training.yaml) settings active during this run:
#   epochs            : 30 (--epochs 30 override)
#   batch_size        : 32 (--batch-size 32 override)
#   device            : cuda
#   mixed_precision   : true  → torch.cuda.amp AMP (automatic mixed precision)
#   optimizer         : AdamW  lr=1e-4  weight_decay=1e-5
#   scheduler         : ReduceLROnPlateau  factor=0.5  patience=5  min_lr=1e-7
#   warmup            : 3 epochs from lr=1e-6
#   loss              : LabelSmoothingCrossEntropy  smoothing=0.1  use_class_weights=true
#   early_stopping    : patience=15  monitor=val_q3_accuracy  mode=max
#   checkpoint        : save_top_k=3  monitor=val_q3_accuracy
#   gradient_clip     : 1.0
#   gradient_accum    : 4 steps
import subprocess, sys, time

cmd = [
    sys.executable, 'train.py',
    '--device', 'cuda',
    '--epochs', '30',
    '--batch-size', '32',
]

print('Running:', ' '.join(cmd))
print('=' * 60)
t0 = time.time()

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()

elapsed = time.time() - t0
if proc.returncode != 0:
    raise RuntimeError(f'train.py exited with code {proc.returncode} — check output above.')

print(f'\n{"=" * 60}')
print(f'✓ Training complete!')
print(f'  Wall-clock time : {elapsed:.0f}s ({elapsed/60:.1f} min)')

In [ ]:
# ── Cell 8: Verify best checkpoint ────────────────────────────────────────────
# Confirms the checkpoint is real (non-empty optimizer state, valid epoch/metrics).
import torch
from pathlib import Path

models_dir = Path('models')
checkpoints = sorted(models_dir.glob('*.pt'), key=lambda f: f.stat().st_mtime, reverse=True)

if not checkpoints:
    raise RuntimeError('No .pt files found in models/ — training may have failed.')

print('Checkpoints found in models/:')
for ck in checkpoints:
    size_mb = ck.stat().st_size / 1e6
    print(f'  {ck.name}  ({size_mb:.1f} MB)')

best_ck = checkpoints[0]
ck_data = torch.load(best_ck, map_location='cpu', weights_only=False)

epoch         = ck_data.get('epoch', 'MISSING')
val_q3        = ck_data.get('val_q3_accuracy', 'MISSING')
val_q8        = ck_data.get('val_q8_accuracy', 'MISSING')
train_loss    = ck_data.get('train_loss', 'MISSING')
opt_state     = ck_data.get('optimizer_state_dict', {})
opt_non_empty = bool(opt_state) and bool(opt_state.get('state', {}))

print(f'\n✓ Best checkpoint : {best_ck.name}')
print(f'  Epoch            : {epoch}')
print(f'  Val Q3 accuracy  : {val_q3}')
print(f'  Val Q8 accuracy  : {val_q8}')
print(f'  Train loss       : {train_loss}')
print(f'  Optimizer state  : {"✓ non-empty" if opt_non_empty else "✗ EMPTY — bootstrap checkpoint?"}')

assert epoch != 'MISSING' and epoch > 0, \
    f'epoch={epoch} — checkpoint looks like bootstrap init, not a trained model'
assert opt_non_empty, \
    'Optimizer state is empty — this is a bootstrap/random-init checkpoint, not a trained one'
assert val_q3 != 'MISSING' and val_q3 > 0.40, \
    f'val_q3_accuracy={val_q3} — too low, check training logs'

print('\n✓ Checkpoint validation passed — this is a real trained checkpoint.')

In [ ]:
# ── Cell 9: Package outputs for download ──────────────────────────────────────
import zipfile
from pathlib import Path

output_zip = Path('/content/protintel_trained_outputs.zip')

with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as z:
    # All model checkpoints
    ck_count = 0
    for ck in Path('models').glob('*.pt'):
        z.write(ck, f'models/{ck.name}')
        print(f'  Added: models/{ck.name}  ({ck.stat().st_size/1e6:.1f} MB)')
        ck_count += 1

    if ck_count == 0:
        raise RuntimeError('No checkpoints to package — training may have failed.')

    # TensorBoard event logs
    log_root = Path('logs')
    log_count = 0
    if log_root.exists():
        for f in log_root.rglob('*'):
            if f.is_file():
                z.write(f, f'logs/{f.relative_to(log_root)}')
                log_count += 1
        print(f'  Added: logs/ ({log_count} files)')

zip_size_mb = output_zip.stat().st_size / 1e6
print(f'\n✓ Output zip: {output_zip}')
print(f'  Size: {zip_size_mb:.1f} MB')
print(f'  Checkpoints: {ck_count}')
print()
print('Next: run Cell 10 to auto-download the zip.')

In [ ]:
# ── Cell 10: Auto-download trained outputs ────────────────────────────────────
# Downloads protintel_trained_outputs.zip directly to your local machine.
# If the download dialog doesn't appear, use Files panel → /content/ → right-click → Download.
from google.colab import files
files.download('/content/protintel_trained_outputs.zip')
print('✓ Download triggered. Check your browser downloads.')